# Week 3 — Responses API, structured outputs, and secure development

Use the project-scoped `/openai/v1/responses` path for new Foundry work. Treat model output as untrusted, validate it at the application boundary, and inventory the model, SDK, prompt, data, and tools used by a release.

In [ ]:
import importlib.util
import sys
from pathlib import Path

from pydantic import BaseModel, ConfigDict, Field

curriculum_root = next(
    candidate
    for base in (Path.cwd(), *Path.cwd().parents)
    for candidate in (base, base / "examples" / "foundry-curriculum")
    if (candidate / "notebook_setup.py").is_file()
)
spec = importlib.util.spec_from_file_location(
    "foundry_curriculum_setup", curriculum_root / "notebook_setup.py"
)
helpers = importlib.util.module_from_spec(spec)
sys.modules[spec.name] = helpers
spec.loader.exec_module(helpers)
session = helpers.load_session(curriculum_root)
session.safe_summary()

In [ ]:
class GroundedAnswer(BaseModel):
    model_config = ConfigDict(extra="forbid", frozen=True)

    answer: str = Field(min_length=1, max_length=2000)
    citations: tuple[str, ...]
    uncertainty: str | None = None


synthetic_candidate = {
    "answer": "The project endpoint scopes access to project capabilities.",
    "citations": ("FOUNDATIONS-001",),
    "uncertainty": None,
}
validated = GroundedAnswer.model_validate(synthetic_candidate)
validated.model_dump()

In [ ]:
release_inventory = {
    "logical_model": session.logical_model,
    "deployment": session.deployment,
    "sdk_family": "OpenAI SDK through aai-core native client",
    "prompt_digest": "record-at-release",
    "dataset_version": "foundry-curriculum-eval-v1",
    "tool_schema_versions": [],
    "dependency_scan_complete": False,
    "output_schema": "GroundedAnswer/v1",
}
release_inventory

## Exit criteria

Exercise valid, missing-field, extra-field, and oversized outputs. Confirm structured-output support for the selected model and scenario rather than assuming it is universal. Add timeouts, bounded retries, safe rendering, and dependency provenance to the release evidence.